Stage 1:
- Read original unittemplate JSON
- Clean and reorder original unittemplate JSON starting top left then counter clockwise
(top-left = largest y value. If same y, choose the smaller x value.)
- Create mirror x, y and o (origin) versions + reorder
- Translate mirrors back to positive axis with origin as close as possible + reorder
- Save as new _mirrors.json file

In [1]:
import json
from pathlib import Path
from copy import deepcopy


# ============================================================
# USER PATHS
# ============================================================

INPUT_JSON = r"C:\Users\Heng_\OneDrive - National University of Singapore\NUS cloud\IHPC\scenarios\tampines JSON\unittemplates_rm.json"
OUTPUT_JSON = r"C:\Users\Heng_\OneDrive - National University of Singapore\NUS cloud\IHPC\scenarios\tampines JSON\unittemplates_rm_mirrors.json"


# ============================================================
# 1. Polygon ordering helpers
# ============================================================

def polygon_signed_area(coords):
    """
    Shoelace formula.

    Positive area = counter-clockwise
    Negative area = clockwise
    """
    area = 0.0
    n = len(coords)

    for i in range(n):
        x1, y1 = coords[i]
        x2, y2 = coords[(i + 1) % n]
        area += x1 * y2 - x2 * y1

    return area / 2.0


def remove_duplicate_consecutive_points(coords):
    """
    Remove repeated neighbouring coordinates.
    Example:
    [[1,1], [2,2], [2,2], [3,3]]
    becomes:
    [[1,1], [2,2], [3,3]]
    """
    cleaned = []

    for p in coords:
        p = [float(p[0]), float(p[1])]

        if not cleaned or cleaned[-1] != p:
            cleaned.append(p)

    # If last point repeats first point, remove last point.
    if len(cleaned) > 1 and cleaned[0] == cleaned[-1]:
        cleaned.pop()

    return cleaned


def ensure_counter_clockwise(coords):
    """
    Make sure the polygon is counter-clockwise.
    """
    coords = remove_duplicate_consecutive_points(coords)

    if polygon_signed_area(coords) < 0:
        coords = coords[::-1]

    return coords


def start_from_top_left(coords):
    """
    Your definition:
    top-left = largest y value.
    If same y, choose the smaller x value.
    """
    coords = [list(p) for p in coords]

    start_i = max(
        range(len(coords)),
        key=lambda i: (coords[i][1], -coords[i][0])
    )

    return coords[start_i:] + coords[:start_i]


def reorder_vertices(coords):
    """
    Final vertex convention:
    1. Remove duplicate consecutive points
    2. Ensure counter-clockwise order
    3. Start from top-left corner
    """
    coords = ensure_counter_clockwise(coords)
    coords = start_from_top_left(coords)
    return coords


# ============================================================
# 2. Mirror helpers
# ============================================================

def mirror_coordinates(coords, mirror_type):
    """
    mirror_x: mirror about x-axis, y -> -y
    mirror_y: mirror about y-axis, x -> -x
    mirror_o: mirror about origin, x -> -x and y -> -y
    """
    mirrored = []

    for x, y in coords:
        if mirror_type == "mirror_x":
            new_x = x
            new_y = -y

        elif mirror_type == "mirror_y":
            new_x = -x
            new_y = y

        elif mirror_type == "mirror_o":
            new_x = -x
            new_y = -y

        else:
            raise ValueError(f"Unknown mirror type: {mirror_type}")

        mirrored.append([new_x, new_y])

    # Reorder again after mirroring
    return reorder_vertices(mirrored)


def mirror_point(point, mirror_type):
    """Mirror one [x, y] point without polygon reordering."""
    x, y = point

    if mirror_type == "mirror_x":
        return [x, -y]
    if mirror_type == "mirror_y":
        return [-x, y]
    if mirror_type == "mirror_o":
        return [-x, -y]

    raise ValueError(f"Unknown mirror type: {mirror_type}")


def transform_facades(facades, point_transform):
    """
    Apply the same geometry transform to every facade.

    Supports:
    - windows:  {"start": [x,y], "end": [x,y], ...}
    - corridors: {"points": [[x,y], ...], ...}
    """
    transformed = deepcopy(facades)

    for facade in transformed:
        if "start" in facade:
            facade["start"] = point_transform(facade["start"])
        if "end" in facade:
            facade["end"] = point_transform(facade["end"])
        if "points" in facade:
            facade["points"] = [point_transform(p) for p in facade["points"]]

    return transformed


# ============================================================
# 3. Translation helpers
# ============================================================

def get_unit_bounds(unit_data):
    """
    Get min/max x and y across all rooms in a unit template.
    """
    all_x = []
    all_y = []

    for room_name, room_data in unit_data["rooms"].items():
        for x, y in room_data["coordinates"]:
            all_x.append(x)
            all_y.append(y)

    return min(all_x), max(all_x), min(all_y), max(all_y)


def translate_unit_to_origin(unit_data):
    """
    Translate the whole unit so:
    min_x = 0
    min_y = 0

    This places the mirrored layout back into a positive coordinate box.
    """
    unit_data = deepcopy(unit_data)

    min_x, max_x, min_y, max_y = get_unit_bounds(unit_data)

    tx = -min_x
    ty = -min_y

    for room_name, room_data in unit_data["rooms"].items():
        translated = []

        for x, y in room_data["coordinates"]:
            translated.append([x + tx, y + ty])

        room_data["coordinates"] = reorder_vertices(translated)

    # Translate window and corridor facade points by the exact same offset.
    if "facades" in unit_data:
        unit_data["facades"] = transform_facades(
            unit_data["facades"],
            lambda p: [p[0] + tx, p[1] + ty]
        )

    return unit_data


# ============================================================
# 4. Template processing helpers
# ============================================================

def clean_original_template(template_data):
    """
    Clean original template:
    - reorder all room coordinates
    - translate to origin just in case
    """
    cleaned = deepcopy(template_data)

    for room_name, room_data in cleaned["rooms"].items():
        room_data["coordinates"] = reorder_vertices(room_data["coordinates"])

    cleaned = translate_unit_to_origin(cleaned)

    return cleaned


def create_mirrored_template(template_data, mirror_type):
    """
    Create one mirrored version of a unit template.
    """
    mirrored = deepcopy(template_data)

    for room_name, room_data in mirrored["rooms"].items():
        original_coords = room_data["coordinates"]
        room_data["coordinates"] = mirror_coordinates(original_coords, mirror_type)

    # Mirror windows and corridor boundaries with the rooms.
    if "facades" in mirrored:
        mirrored["facades"] = transform_facades(
            mirrored["facades"],
            lambda p: mirror_point(p, mirror_type)
        )

    mirrored = translate_unit_to_origin(mirrored)

    return mirrored

# ============================================================
# 5. Custom compact JSON writer
# ============================================================

def format_value(value):
    """
    Format numbers nicely:
    8.0 -> 8.0
    0.0 -> 0.0
    keep floats clean
    """
    if isinstance(value, int):
        return str(value)

    if isinstance(value, float):
        return str(round(value, 6))

    if isinstance(value, str):
        return json.dumps(value)

    return json.dumps(value)


def format_coordinates(coords):
    """
    Convert coordinates into compact form:
    [[x1,y1],[x2,y2],[x3,y3]]
    """
    coord_strings = []

    for x, y in coords:
        coord_strings.append(
            f"[{format_value(x)},{format_value(y)}]"
        )

    return "[" + ",".join(coord_strings) + "]"


def format_json_compact(value):
    """Compact JSON for facade properties and coordinate arrays."""
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def write_compact_unittemplates(data, output_path):
    """
    Write unit template JSON in your preferred compact format.

    Example output:

    "5r_corr_end": {
    "rooms": {
        "masterroom": {"coordinates": [[...],[...]]},
        "bedroom2": {"coordinates": [[...],[...]]}
    },
    "window": {"windowFacing": "corridor","WWR": 0.9}
    },
    """

    output_path = Path(output_path)

    lines = []
    lines.append("{")

    template_items = list(data.items())

    for template_i, (template_name, template_data) in enumerate(template_items):

        lines.append(f'    "{template_name}": {{')
        lines.append(f'    "rooms": {{')

        room_items = list(template_data["rooms"].items())

        for room_i, (room_name, room_data) in enumerate(room_items):
            coords_str = format_coordinates(room_data["coordinates"])

            comma = "," if room_i < len(room_items) - 1 else ""

            line = (
                f'        "{room_name}": '
                f'{{"coordinates": {coords_str}}}'
                f'{comma}'
            )

            lines.append(line)

        lines.append("    },")

        template_comma = "," if template_i < len(template_items) - 1 else ""

        # Preserve the new explicit window/corridor facade geometry.
        if "facades" in template_data:
            facades_str = format_json_compact(template_data["facades"])
            lines.append(f'    "facades": {facades_str}')
        # Backward compatibility for templates using the old window metadata.
        elif "window" in template_data:
            window_str = format_json_compact(template_data["window"])
            lines.append(f'    "window": {window_str}')
        else:
            # Remove the trailing comma after the rooms object when no
            # additional template-level fields exist.
            lines[-1] = "    }"

        lines.append(f'    }}{template_comma}')
        lines.append("")

    lines.append("}")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"Saved compact JSON to: {output_path}")
    
# ============================================================
# Main Stage 1 function
# ============================================================

def generate_stage1_origined_templates(input_json, output_json):
    """
    Replicates Stage 1:

    1. Load original unit template JSON
    2. Reorder vertices:
       - start from top-left
       - counter-clockwise
    3. Mirror each unit about:
       - x-axis
       - y-axis
       - origin
    4. Translate every final unit to positive coordinates
    5. Save final origined JSON
    """
    input_json = Path(input_json)
    output_json = Path(output_json)

    with open(input_json, "r", encoding="utf-8") as f:
        original_templates = json.load(f)

    final_templates = {}

    for template_name, template_data in original_templates.items():

        # -------------------------------
        # A. Original cleaned version
        # -------------------------------
        cleaned_original = clean_original_template(template_data)
        final_templates[template_name] = cleaned_original

        # -------------------------------
        # B. Mirrored versions
        # -------------------------------
        for mirror_type in ["mirror_x", "mirror_y", "mirror_o"]:

            mirrored_template = create_mirrored_template(
                template_data=template_data,
                mirror_type=mirror_type
            )

            new_name = f"{template_name}_{mirror_type}"
            final_templates[new_name] = mirrored_template

    write_compact_unittemplates(final_templates, output_json)

    print("Stage 1 completed.")
    print(f"Input file:  {input_json}")
    print(f"Output file: {output_json}")
    print(f"Original templates: {len(original_templates)}")
    print(f"Final templates:    {len(final_templates)}")

    return final_templates

# ============================================================
# Run
# ============================================================

templates = generate_stage1_origined_templates(
    input_json=INPUT_JSON,
    output_json=OUTPUT_JSON
)

Saved compact JSON to: C:\Users\Heng_\OneDrive - National University of Singapore\NUS cloud\IHPC\scenarios\tampines JSON\unittemplates_rm_mirrors.json
Stage 1 completed.
Input file:  C:\Users\Heng_\OneDrive - National University of Singapore\NUS cloud\IHPC\scenarios\tampines JSON\unittemplates_rm.json
Output file: C:\Users\Heng_\OneDrive - National University of Singapore\NUS cloud\IHPC\scenarios\tampines JSON\unittemplates_rm_mirrors.json
Original templates: 5
Final templates:    20
